In [1]:
import sys
sys.dont_write_bytecode = True
from _init_notebook import *
sys.dont_write_bytecode = False

Importes inicializado.
Project root es: /Users/federico/Workspace/correlation-notebooks
Entorno inicializado.


In [2]:
BASE_PATH = "../data/yahoo_1d"
loaded_pct_changes = []

def normalize_symbol(symbol: str) -> str:
    # Reemplaza caracteres problemáticos para que coincidan con archivos
    return symbol.replace("^", "").replace("/", "-")

# Excluimos criptos por operar los fines de semana
for group, assets in SYMBOL_GROUPS_YAHOO.items():
    if group == "cripto":
        continue

    for symbol in assets:
        normalized_symbol = normalize_symbol(symbol)
        file_path = os.path.join(BASE_PATH, group, f"{normalized_symbol}.csv")
        if not os.path.exists(file_path):
            print(f"Archivo no encontrado: {file_path}")
            continue

        try:
            df = pd.read_csv(file_path, skiprows=[1])
            df.columns = ['Date', f'{symbol}.Close', f'{symbol}.High', f'{symbol}.Low',
                          f'{symbol}.Open', f'{symbol}.Volume']
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.set_index('Date').sort_index()

            df = calc_pct_change(df, name=symbol)
            pct_col = df[[f'{symbol}.PctChange']]
            loaded_pct_changes.append(pct_col)

        except Exception as e:
            print(f"Error procesando {symbol}: {e}")

# Combinar todo por fecha
if loaded_pct_changes:
    pct_change_matrix = pd.concat(loaded_pct_changes, axis=1).dropna()
    #print("Columnas en la matriz de pct_change:")
    #print(pct_change_matrix.columns)
    #print(pct_change_matrix.head())
else:
    print("No se cargaron datos.")

Archivo no encontrado: ../data/yahoo_1d/forex/EURUSD=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/USDJPY=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/GBPUSD=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/AUDUSD=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/USDCAD=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/USDCHF=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/NZDUSD=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/EURJPY=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/EURGBP=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/CHFJPY=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/AUDJPY=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/EURAUD=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/GBPJPY=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/GBPAUD=X.csv
Archivo no encontrado: ../data/yahoo_1d/forex/AUDNZD=X.csv


In [3]:
len(pct_change_matrix.columns)

70

In [4]:
# Fecha límite para truncar
fecha_limite = '2013-01-04'

# Truncamos el DataFrame desde esa fecha inclusive
pct_change_truncated = pct_change_matrix.loc[fecha_limite:].copy()

# Verificamos si quedan valores NaN en el DataFrame truncado
nan_por_columna = pct_change_truncated.isnull().sum()

# Imprimimos resumen
print(f"Cantidad de filas después de truncar: {len(pct_change_truncated)}")
print("Valores NaN por columna después de truncar:")
print(nan_por_columna[nan_por_columna > 0])

Cantidad de filas después de truncar: 2621
Valores NaN por columna después de truncar:
Series([], dtype: int64)


In [ ]:
df_resultado = compute_pairwise_correlations_incremental(pct_change_truncated, max_lag=180, output_path="correlation_results_with_lags.csv", resume=True)
print(df_resultado.head())

  0%|          | 0/2415 [00:00<?, ?it/s]